# 1.1.1 Setup & Environment

This notebook walks through the essential steps for setting up a Python environment
for machine learning:

1. Verify Python version & platform
2. Check / install required ML packages
3. Detect GPU availability
4. Download a real dataset from Hugging Face Hub
5. Quick EDA and training sanity-check
6. Save a reproducibility artifact

**Run each cell in order.**

In [ ]:
import sys
import platform
import os

print('Python  :', sys.version)
print('Platform:', platform.platform())
print('Prefix  :', sys.prefix)
in_venv = sys.prefix != sys.base_prefix
print('In venv :', in_venv)

In [ ]:
# ── Install / verify core packages ──────────────────────────────────────────
import importlib, subprocess

PACKAGES = {
    'numpy':      'numpy',
    'pandas':     'pandas',
    'matplotlib': 'matplotlib',
    'sklearn':    'scikit-learn',
    'scipy':      'scipy',
}

for import_name, pip_name in PACKAGES.items():
    try:
        mod = importlib.import_module(import_name)
        print(f'  ✓  {pip_name:<20}  v{getattr(mod, "__version__", "?")}')
    except ImportError:
        print(f'  ⬇  Installing {pip_name} …')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])
        print(f'  ✓  {pip_name} installed')

In [ ]:
# ── GPU detection ────────────────────────────────────────────────────────────
try:
    import torch
    print('PyTorch version :', torch.__version__)
    print('CUDA available  :', torch.cuda.is_available())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}: {p.name}  {p.total_memory // 1024**2} MB')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print('Apple MPS available')
except ImportError:
    print('PyTorch not installed — pip install torch')

In [ ]:
# ── Download Iris dataset from Hugging Face Hub ──────────────────────────────
from pathlib import Path
import urllib.request

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
dest = DATA_DIR / 'iris.csv'

if not dest.exists():
    url = 'https://huggingface.co/datasets/scikit-learn/iris/resolve/main/Iris.csv'
    print(f'Downloading from {url} …')
    try:
        urllib.request.urlretrieve(url, dest)
        print(f'Saved to {dest}  ({dest.stat().st_size} bytes)')
    except Exception as e:
        print(f'Download failed: {e}. Will use sklearn built-in dataset.')
        dest = None
else:
    print(f'Using cached file: {dest}')

In [ ]:
# ── Load and explore the dataset ─────────────────────────────────────────────
import pandas as pd
import numpy as np

if dest and Path(dest).exists():
    df = pd.read_csv(dest)
else:
    from sklearn.datasets import load_iris
    iris = load_iris(as_frame=True)
    df = iris.frame
    df.columns = [c.replace(' (cm)', '').replace(' ', '_') for c in df.columns]

print(df.head())
print(f'\nShape: {df.shape}')
df.describe()

In [ ]:
# ── Visualise ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

numeric = df.select_dtypes('number')
species_col = 'Species' if 'Species' in df.columns else 'target'

fig, axes = plt.subplots(1, len(numeric.columns), figsize=(14, 3))
for ax, col in zip(axes, numeric.columns):
    ax.hist(numeric[col], bins=20, edgecolor='black')
    ax.set_title(col)
plt.suptitle('Feature Distributions — Iris Dataset', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Train a quick model ───────────────────────────────────────────────────────
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

feature_cols = [c for c in numeric.columns]
X = df[feature_cols].values
y = LabelEncoder().fit_transform(df[species_col]) if df[species_col].dtype == object else df[species_col].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in [
    ('LogisticRegression', LogisticRegression(max_iter=500)),
    ('RandomForest',       RandomForestClassifier(n_estimators=100, random_state=42)),
]:
    pipe   = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    print(f'{name:<25}  acc = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# ── Save environment fingerprint ─────────────────────────────────────────────
import json, hashlib

env = {
    'python': sys.version,
    'platform': platform.platform(),
    'packages': {
        pkg: getattr(importlib.import_module(pkg), '__version__', '?')
        for pkg in ['numpy', 'pandas', 'sklearn', 'scipy', 'matplotlib']
        if importlib.util.find_spec(pkg)
    },
}
fingerprint = hashlib.sha256(json.dumps(env, sort_keys=True).encode()).hexdigest()[:12]
print(f'Environment fingerprint: {fingerprint}')
print(json.dumps(env, indent=2))

## Summary

| Concept | Tool |
|---------|------|
| Virtual environments | `venv`, `conda`, `uv` |
| Package management | `pip`, `uv`, `conda` |
| Modern project setup | `pyproject.toml` |
| Reproducibility | environment fingerprint + `requirements.txt` / `uv.lock` |
| GPU detection | `torch.cuda.is_available()` |

**Next:** `1.1.2 Basic Syntax & Data Types`